In [0]:
#

apple_df = spark.table('hkdatabricks.datasets.apple_global_sales_dataset')

In [0]:
apple_df.display()

In [0]:
apple_df.createOrReplaceTempView('apple_sales')


In [0]:
%sql

select * from apple_sales;

In [0]:
%sql

select * from apple_sales where country = "India";

In [0]:
%sql

select count(*) as Total_Sales from apple_sales where country = "India";

In [0]:
# How to do above condition in spark
from pyspark.sql import functions as F
# By using Filter
india_df = apple_df.filter(F.col('country')== "India")
india_df.display()

In [0]:
india_df.count()

In [0]:
# We can use both filter and Where
india_df = apple_df.where(F.col('country')== "India")
india_df.display()


In [0]:
# Checking null values 
null_rtg_df = apple_df.filter(F.col("customer_rating").isNull())

null_rtg_df.count()

In [0]:
# Checking not null values 
null_rtg_df = apple_df.filter(F.col("customer_rating").isNotNull())

null_rtg_df.count()

In [0]:
# check both null and not null
apple_df.select(
    F.count(F.when(F.col("customer_rating").isNull(), 1)).alias("null_count"),
    
    F.count(F.when(F.col("customer_rating").isNotNull(), 1)).alias("not_null_count")
).show()

In [0]:
apple_df.select("customer_rating").count()

In [0]:
# Drop & drop Dublicates columns
apple_df = apple_df.drop('customer_segment','payment_method')

apple_df.columns


In [0]:
# Distinct
apple_df.distinct().count()

In [0]:
revenue_df = (
    apple_df.groupBy("country").
    agg(
        F.sum(F.col("units_sold")).alias("Total_Units_Sold"),
        F.ceil(F.sum(F.col("revenue_usd")).alias("Total_Revenue_USd"))

    )
)

revenue_df.display()

In [0]:
 # Q.) In a Given Year I want to find how many products where kept and how many were Exchanged and How many were Returned
# We can also add Quater
df_Q = (apple_df
      .groupBy("year", "return_status","quarter")
      .agg(
            F.count('*').alias("Count_All")
        )
    .orderBy("year")
)

df_Q.display()





In [0]:
%sql
select * from apple_sales;


In [0]:
# Monthly Report for Revenue all years of each month 
df1 = (
    apple_df
    .groupBy("year","month")
    .agg(
        F.ceil(F.sum(F.col("revenue_usd")).alias("Total_Revenue_USd"))
    )
    .orderBy("month")
)

df1.display()
# Monthly Report for Revenu all years


In [0]:

# Monthly Report for Revenue all years of each month 
# Extracting using sales Date
df2 = (
    apple_df.
    groupBy(F.year("sale_date").alias("year"),F.month("sale_date").alias("month"))
    .agg(
        F.round(F.sum(F.col("revenue_usd")).alias("Total_Revenue_USd"))
    )
    .orderBy(F.year("sale_date"),F.month("sale_date"))
)

df2.display()
